# Experiment: AMPK NRF2 Expansion Gate

Objective: decide how the `AMPKB1` / `NRF2` lane should enter the next lab plan without becoming a patient-treatment claim.

Success criteria:
- separate direct erythroid HbF evidence from generic AMPK/NRF2 biology;
- keep metformin as a translational warning if human thalassemia evidence is weak or negative;
- return a promote, hold, or reject label for each expansion item.


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

ROOT = Path.cwd()
DATA_DIR = "data/literature/pubmed"
SOURCE_FILES = {
    "metformin": f"{DATA_DIR}/2026-04-27-metformin-hbf-thalassemia-search.json",
    "ampkb1_refresh": f"{DATA_DIR}/2026-04-27-ampkb1-nrf2-ulk1-hbf-refresh.json",
    "nrf2": f"{DATA_DIR}/2026-04-27-nrf2-hbf-beta-thalassemia-search.json",
    "pf_06409577": f"{DATA_DIR}/2026-04-27-pf-06409577-ampk-search.json",
}

## Plan

This is a computational evidence-gating experiment, not wet-lab evidence.

Hypothesis: `AMPKB1` / `NRF2` is valuable for assay design, but the near-term candidate role should be `pathway_probe`, not `therapy_candidate`.

Decision rule:
- direct erythroid HbF plus beta-thalassemia relevance can move a lane to `assay_probe`;
- human thalassemia evidence with no added benefit moves a lane to `negative_or_low_priority_comparator`;
- broad pathway evidence without beta-thalassemia validation stays `mechanism_context_only`.


In [ ]:
@dataclass(frozen=True)
class GateInput:
    """One AMPK/NRF2 expansion item and its evidence features."""

    name: str
    direct_hbf: bool
    beta_thal_context: bool
    human_thal_context: bool
    negative_human_signal: bool
    exact_probe_identity: bool
    broad_pathway_risk: bool
    source_key: str
    boundary: str


def load_snapshot(key: str) -> dict[str, object]:
    """Load one PubMed snapshot by source key."""
    path = ROOT / SOURCE_FILES[key]
    return json.loads(path.read_text(encoding="utf-8"))


def snapshot_titles(key: str) -> list[str]:
    """Return compact result titles for one source snapshot."""
    snapshot = load_snapshot(key)
    results = snapshot.get("results", [])
    if not isinstance(results, list):
        return []
    return [str(result.get("title", "")) for result in results]

In [ ]:
items = [
    GateInput(
        name="selective AMPKB1 / PF-06409577-style probe",
        direct_hbf=True,
        beta_thal_context=False,
        human_thal_context=False,
        negative_human_signal=False,
        exact_probe_identity=True,
        broad_pathway_risk=True,
        source_key="pf_06409577",
        boundary=(
            "pathway probe only; SCD-centered HbF evidence needs "
            "beta-thalassemia validation"
        ),
    ),
    GateInput(
        name="metformin",
        direct_hbf=True,
        beta_thal_context=True,
        human_thal_context=True,
        negative_human_signal=True,
        exact_probe_identity=True,
        broad_pathway_risk=True,
        source_key="metformin",
        boundary=(
            "translational warning comparator; NTDT study reported no "
            "extra benefit over hydroxyurea"
        ),
    ),
    GateInput(
        name="generic NRF2 activation",
        direct_hbf=True,
        beta_thal_context=True,
        human_thal_context=False,
        negative_human_signal=False,
        exact_probe_identity=False,
        broad_pathway_risk=True,
        source_key="nrf2",
        boundary="mechanism context only unless a defined probe passes safety gates",
    ),
]

In [ ]:
def classify_item(item: GateInput) -> str:
    """Classify one item for the AMPK/NRF2 expansion lane."""
    if item.negative_human_signal:
        return "negative_or_low_priority_comparator"

    if item.direct_hbf and item.exact_probe_identity and not item.human_thal_context:
        return "assay_probe_only"

    if item.direct_hbf and item.beta_thal_context and not item.exact_probe_identity:
        return "mechanism_context_only"

    return "hold"


def summarize_item(item: GateInput) -> dict[str, object]:
    """Return one compact gate-result row."""
    return {
        "item": item.name,
        "label": classify_item(item),
        "source_count": load_snapshot(item.source_key).get("count", 0),
        "first_source_title": snapshot_titles(item.source_key)[:1],
        "boundary": item.boundary,
    }

In [ ]:
gate_results = [summarize_item(item) for item in items]

decision = {
    "lane": "AMPKB1_NRF2_expansion",
    "decision": "assay_probe_not_patient_candidate",
    "assay_probe_items": [
        row["item"] for row in gate_results if row["label"] == "assay_probe_only"
    ],
    "negative_or_low_priority": [
        row["item"]
        for row in gate_results
        if row["label"] == "negative_or_low_priority_comparator"
    ],
    "mechanism_context": [
        row["item"] for row in gate_results if row["label"] == "mechanism_context_only"
    ],
}

decision

In [ ]:
gate_results

## Results

The expansion gate keeps `AMPKB1` / `NRF2` as a lab-probe lane, not a therapy lane.

- A selective `AMPKB1` / PF-06409577-style probe is useful if the lab can source a defined research-grade compound and measure HbF, `NRF2`, phospho-`ULK1`, p62 / `SQSTM1`, alpha-globin burden, maturation, and hemolysis.
- Metformin is not promoted. It becomes a translational warning comparator because the retrieved NTDT human study did not show extra benefit when added to stable hydroxyurea.
- Generic `NRF2` activation remains mechanism context only. A broad antioxidant-response signal is not enough without a defined probe and safety gates.


## Interpretation

The useful idea is not “activate AMPK in a patient.” The useful idea is to ask a qualified lab whether a defined `AMPKB1` / noncanonical `NRF2` probe can produce a dual signal: HbF induction plus alpha-globin cleanup without stress toxicity.

This lane should be rejected or held if the signal depends only on K562, broad oxidative stress, missing alpha-globin readouts, missing maturation markers, or a compound identity that cannot be verified.
